<a href="https://colab.research.google.com/github/VISHWAJA-028/ML_Multimodality/blob/main/build_bundly_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
%%writefile /content/build_bundle.py

Writing /content/build_bundle.py


In [13]:
%run /content/build_bundle.py

In [15]:
import os

print("Checking artifacts folder...\n")

art = "/content/drive/MyDrive/PCOS/artifacts"

if os.path.exists(art):
    print("✅ Artifacts folder exists\n")
    print("Files currently there:")

    for f in os.listdir(art):
        path = os.path.join(art, f)
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f}  →  {size:.1f} MB")
else:
    print("❌ Artifacts folder does not exist")

Checking artifacts folder...

✅ Artifacts folder exists

Files currently there:
  structured_model_comparison.csv  →  0.0 MB
  structured_model_info.json  →  0.0 MB
  structured_imputer.joblib  →  0.0 MB
  structured_columns.joblib  →  0.0 MB
  structured_scaler.joblib  →  0.0 MB
  structured_model.joblib  →  1.0 MB
  __pycache__  →  0.0 MB
  structured_bundle.joblib  →  1.0 MB
  PCOS_best_image_model.keras  →  202.2 MB
  PCOS_image_model_info.json  →  0.0 MB
  model_comparison.csv  →  0.0 MB


In [16]:
%run /content/build_bundle.py

In [17]:
import os

path = "/content/build_bundle.py"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Size:", os.path.getsize(path), "bytes")

    with open(path, "r") as f:
        content = f.read()

    print("\nFirst 500 characters:\n")
    print(content[:500])

Exists: True
Size: 1 bytes

First 500 characters:

 


In [18]:
%%writefile /content/build_bundle.py
import json
import os
import joblib

from pcos_predictor import PCOSPredictor, create_bundle

ART = "/content/drive/MyDrive/PCOS/artifacts"
OUT = os.path.join(ART, "pcos_bundle.pkl")

print("Starting bundle creation...")

required = [
    "structured_model.joblib",
    "structured_scaler.joblib",
    "structured_imputer.joblib",
    "structured_columns.joblib",
    "PCOS_best_image_model.keras",
    "PCOS_image_model_info.json"
]

for filename in required:
    path = os.path.join(ART, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

    print("✅ Found:", filename)

print("\nLoading structured model...")

tab_model = joblib.load(
    os.path.join(ART, "structured_model.joblib")
)

tab_scaler = joblib.load(
    os.path.join(ART, "structured_scaler.joblib")
)

tab_imputer = joblib.load(
    os.path.join(ART, "structured_imputer.joblib")
)

tab_columns = joblib.load(
    os.path.join(ART, "structured_columns.joblib")
)

print("✅ Structured artifacts loaded")

with open(
    os.path.join(ART, "PCOS_image_model_info.json"),
    "r"
) as f:
    info = json.load(f)

print("\nImage model:", info.get("best_model"))
print("Image size:", info.get("image_size"))

print("\nCreating bundle...")

create_bundle(
    OUT,
    tab_model=tab_model,
    tab_imputer=tab_imputer,
    tab_scaler=tab_scaler,
    tab_columns=tab_columns,
    keras_model_path=os.path.join(
        ART,
        "PCOS_best_image_model.keras"
    ),
    img_size=tuple(
        info.get("image_size", [224, 224])
    ),
    image_p_is_pcos=info.get("p_is_pcos", False),
    image_threshold=info.get("pcos_threshold", 0.5),
    image_tta=info.get("tta", 1),
    meta={
        "image_backbone": info.get("best_model"),
        "image_test_metrics": {
            k: info.get(k)
            for k in (
                "accuracy",
                "precision",
                "recall",
                "f1_score",
                "roc_auc"
            )
        }
    }
)

print("\n================================")
print("✅ BUNDLE CREATED SUCCESSFULLY")
print("================================")
print("Location:", OUT)
print(
    "Size:",
    round(os.path.getsize(OUT) / (1024 * 1024), 1),
    "MB"
)

Overwriting /content/build_bundle.py


In [19]:
import os

path = "/content/build_bundle.py"

print("Size:", os.path.getsize(path), "bytes")

Size: 2238 bytes


In [20]:
%run /content/build_bundle.py

Starting bundle creation...
✅ Found: structured_model.joblib
✅ Found: structured_scaler.joblib
✅ Found: structured_imputer.joblib
✅ Found: structured_columns.joblib
✅ Found: PCOS_best_image_model.keras
✅ Found: PCOS_image_model_info.json

Loading structured model...
✅ Structured artifacts loaded

Image model: ResNet50
Image size: [224, 224]

Creating bundle...

✅ BUNDLE CREATED SUCCESSFULLY
Location: /content/drive/MyDrive/PCOS/artifacts/pcos_bundle.pkl
Size: 203.3 MB


In [21]:
from pcos_predictor import PCOSPredictor

bundle_path = "/content/drive/MyDrive/PCOS/artifacts/pcos_bundle.pkl"

predictor = PCOSPredictor.load(bundle_path)

print("✅ Multimodal predictor loaded successfully!")
predictor.describe()

✅ Multimodal predictor loaded successfully!
Structured columns: 58 | threshold 0.5
Image size: (224, 224) | P is PCOS: False | threshold: 0.5 | TTA: 1
Meta: {'image_backbone': 'ResNet50', 'image_test_metrics': {'accuracy': 0.8721719457013575, 'precision': 0.7684426229508197, 'recall': 1.0, 'f1_score': 0.8690614136732329, 'roc_auc': 0.9989829731499673}}
Saved with: {'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'lightgbm': '4.6.0', 'xgboost': '3.4.1', 'tensorflow': '2.20.0'}
